# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens"
)
LAYERS = [12, 23, 25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "google/gemma-3-1b-it"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [4]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [5]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                                 \
                base                    base_inv                 ft   
0     The (2.06e-03)                Ⴊ (1.86e-04)     The (3.40e-03)   
1       I (1.70e-03)                 (1.80e-04)    This (2.35e-03)   
2     The (1.65e-03)      recommendet (1.70e-04)       I (2.35e-03)   
3    This (1.60e-03)                𒉪 (1.64e-04)      It (1.95e-03)   
4    This (1.33e-03)      <unused535> (1.64e-04)      In (1.88e-03)   
5      It (1.29e-03)                ꗚ (1.59e-04)     The (1.77e-03)   
6      In (1.17e-03)   exposureButton (1.59e-04)    This (1.34e-03)   
7       I (1.10e-03)   FBSDKBridgeAPI (1.54e-04)      We (1.25e-03)   
8     the (9.99e-04)       <unused27> (1.54e-04)       I (9.19e-04)   
9      We (9.12e-04)        asaddassa (1.54e-04)       是 (9.19e-04)   
10      是 (9.12e-04)          urupani (1.54e-04)      It (8.62e-04)   
11     It (8.58e-04)          suddhak (1.54e-04)   There (8.35e-04)   
12    ' ' (7.32e-04)                ꗏ (1.54e-04)     the (8.09e-04)   
13   '  ' (7.10e-04)                𒐄 (1.50e-04)     You (7.59e-04)   
14   Read (6.87e-04)                𒋰 (1.50e-04)      In (6.71e-04)   
15     In (6.26e-04)                𒑱 (1.50e-04)     ' ' (6.29e-04)   
16     We (6.26e-04)         VSScript (1.50e-04)   There (5.91e-04)   
17  There (5.61e-04)     StarSrvGroup (1.50e-04)     For (5.76e-04)   
18      当 (5.61e-04)        vuccatiti (1.50e-04)       1 (5.76e-04)   
19    You (5.61e-04)      specialmeal (1.50e-04)      We (5.76e-04)   

                                                                             \
                       ft_inv                  diff                diff_inv   
0                Ⴊ (2.10e-04)      assanam (0.2324)            and (0.9844)   
1      <unused535> (2.03e-04)         Firm (0.1807)             or (0.0159)   
2                 (2.03e-04)    Visualize (0.0854)           ** (2.40e-05)   
3      recommendet (1.91e-04)       Injury (0.0664)            ( (5.33e-06)   
4       <unused27> (1.63e-04)           Hd (0.0664)            * (3.67e-06)   
5            راجسټ (1.63e-04)        Women (0.0664)          إلا (3.23e-06)   
6            SRPCS (1.58e-04)     vehement (0.0518)   especially (3.23e-06)   
7      <unused132> (1.58e-04)            를 (0.0403)      getting (1.53e-06)   
8                Ⴚ (1.58e-04)         avkh (0.0315)           Aí (1.05e-06)   
9        <unused7> (1.58e-04)  последствии (0.0315)    activités (9.28e-07)   
10                (1.54e-04)           Em (0.0190)            / (6.37e-07)   
11     <unused970> (1.54e-04)           To (0.0148)           et (4.95e-07)   
12               𒉪 (1.54e-04)      attiyam (0.0116)           (€ (3.87e-07)   
13      <unused53> (1.54e-04)   penchant (7.02e-03)   activities (3.41e-07)   
14        SrvGroup (1.54e-04)    Perhaps (7.02e-03)     examples (3.41e-07)   
15  kloudformation (1.49e-04)         Se (7.02e-03)          but (3.02e-07)   
16               (1.49e-04)          ㎛ (5.46e-03)        using (3.02e-07)   
17               ꗏ (1.49e-04)       Ibid (5.46e-03)         plus (2.66e-07)   
18      ArtStudent (1.49e-04)   Although (5.46e-03)           مع (2.66e-07)   
19    <unused2141> (1.44e-04)        என் (4.24e-03)           aí (2.66e-07)   

                 layer_23                                                    \
                     base                   base_inv                     ft   
0           Okay (0.8086)       opencamer (2.59e-04)          Okay (0.5625)   
1            Let (0.0486)               𒆝 (2.44e-04)           Let (0.0918)   
2           This (0.0190)          Ioctrl (2.44e-04)          This (0.0554)   
3             It (0.0115)  Polynucleaires (2.44e-04)           The (0.0337)   
4             ** (0.0115)    StarSrvGroup (2.29e-04)          Here (0.0204)   
5           Okay (0.0115)    <unused5944> (2.29e-04)            It (0.0150)   
6          The (9.58e-03)         testHDR (2.29e-04)            ** (0.0109)   

In [6]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                                \
                base                base_inv                    ft   
0     the (2.02e-04)         Ibid (7.39e-05)        the (1.83e-04)   
1     een (1.34e-04)          spp (6.53e-05)        een (1.16e-04)   
2      in (1.22e-04)           Kg (5.67e-05)          a (1.03e-04)   
3       a (1.19e-04)            및 (5.17e-05)         in (1.03e-04)   
4      في (1.05e-04)      Whereas (4.70e-05)       more (9.06e-05)   
5     ایک (9.82e-05)          lbs (4.63e-05)       give (9.06e-05)   
6      أن (9.68e-05)         ibid (4.48e-05)         یک (8.39e-05)   
7      یک (9.39e-05)        silam (4.03e-05)        ایک (8.39e-05)   
8    give (9.25e-05)          mbh (3.96e-05)         في (8.25e-05)   
9    more (9.11e-05)           kg (3.96e-05)         أن (8.11e-05)   
10    ' ' (9.11e-05)         Ibid (3.91e-05)        one (8.01e-05)   
11     to (9.11e-05)        riots (3.91e-05)       make (7.72e-05)   
12     în (8.30e-05)   prostitute (3.84e-05)         to (7.49e-05)   
13   make (8.15e-05)          Etc (3.79e-05)         în (7.06e-05)   
14    بال (8.15e-05)         idem (3.72e-05)   somewhat (7.06e-05)   
15    for (8.01e-05)           ®. (3.72e-05)        ' ' (6.96e-05)   
16     من (8.01e-05)     existent (3.72e-05)          ّ (6.96e-05)   
17    one (8.01e-05)   dependents (3.67e-05)       find (6.82e-05)   
18      ف (7.77e-05)    restantes (3.67e-05)          な (6.82e-05)   
19      ّ (7.44e-05)      dDevice (3.60e-05)       take (6.63e-05)   

                                                                               \
                   ft_inv                       diff                 diff_inv   
0          spp (7.53e-05)        mathematic (0.0977)            '  ' (0.6445)   
1         Ibid (5.96e-05)          vehement (0.0280)             etc (0.1118)   
2           Kg (5.41e-05)            cogniz (0.0170)              بت (0.0679)   
3          Etc (4.94e-05)      exorbitant (4.85e-03)               / (0.0466)   
4            및 (4.36e-05)               𒆝 (3.78e-03)              ** (0.0466)   
5           kg (4.29e-05)              zq (3.78e-03)               � (0.0118)   
6          lbs (4.22e-05)      bicyclists (2.94e-03)             ... (0.0104)   
7         ibid (4.10e-05)          urnama (2.29e-03)             и (9.16e-03)   
8           ®. (4.03e-05)               𒂀 (1.79e-03)           and (8.12e-03)   
9      Whereas (3.96e-05)               𒁣 (1.79e-03)          хоро (7.14e-03)   
10     illetve (3.91e-05)          olefin (1.40e-03)            مع (7.14e-03)   
11   restantes (3.91e-05)              미국 (1.40e-03)            ** (5.22e-03)   
12        idem (3.79e-05)           HtIdx (1.40e-03)             ¡ (3.17e-03)   
13       riots (3.79e-05)               𒅼 (1.40e-03)         fique (2.79e-03)   
14     dDevice (3.72e-05)  Polynucleaires (1.40e-03)            or (2.32e-03)   
15         mbh (3.72e-05)       cognizant (1.40e-03)            اس (2.04e-03)   
16          /- (3.67e-05)               𒉕 (1.40e-03)   обязательно (1.50e-03)   
17      ethnic (3.55e-05)         assanam (1.40e-03)    <unused62> (1.50e-03)   
18         eds (3.50e-05)               𒅊 (1.08e-03)          Боль (8.01e-04)   
19     earners (3.50e-05)               𒅾 (1.08e-03)             د (7.51e-04)   

             layer_23                                                \
                 base                   base_inv                 ft   
0        ' ' (0.0820)              渦柱 (2.38e-04)       ' ' (0.0889)   
1          , (0.0547)           HtIdx (2.38e-04)         , (0.0432)   
2        and (0.0400)               𒆝 (2.38e-04)       and (0.0254)   
3          ( (0.0275)          Ioctrl (2.38e-04)        in (0.0225)   
4          a (0.0250)          VSTYPE (2.24e-04)         a (0.0217)   
5         in (0.0221)               ꗕ (2.24e-04)         - (0.0217)   
6          - (0.0214)               ꗮ (2.24e-04)         ( (0.0186)   
7       '  ' (0.0214)               ꗫ (2.24e-04) 

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [7]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                   \
                base                     base_inv                  ft   
0     the (3.47e-04)                 및 (3.93e-05)      the (2.97e-04)   
1     ' ' (2.37e-04)              Ibid (3.84e-05)        a (1.75e-04)   
2       a (2.13e-04)                 疃 (3.78e-05)      ' ' (1.72e-04)   
3      in (1.85e-04)            تباينه (3.49e-05)       in (1.50e-04)   
4      to (1.46e-04)              డాది (3.39e-05)       to (1.18e-04)   
5      في (1.25e-04)                ®. (3.33e-05)      one (1.15e-04)   
6     one (1.23e-04)            andRow (3.30e-05)     more (1.10e-04)   
7     for (1.16e-04)              ibid (3.26e-05)     give (1.04e-04)   
8     een (1.16e-04)           dDevice (3.16e-05)      een (9.91e-05)   
9    more (1.13e-04)               および (3.15e-05)       في (9.80e-05)   
10   give (1.09e-04)               spp (3.07e-05)      for (8.99e-05)   
11     on (1.05e-04)          foresaid (3.02e-05)       on (8.95e-05)   
12   that (1.04e-04)              ไหร่ (2.97e-05)     that (8.62e-05)   
13      ف (1.00e-04)        prostitute (2.95e-05)      the (7.61e-05)   
14     لا (9.40e-05)               eds (2.89e-05)       at (6.95e-05)   
15    بال (9.28e-05)            namani (2.88e-05)        ف (6.86e-05)   
16     من (8.31e-05)               mbh (2.88e-05)       لا (6.86e-05)   
17      ل (8.14e-05)              ccgi (2.84e-05)   choose (6.80e-05)   
18     as (8.01e-05)              elwv (2.79e-05)       as (6.57e-05)   
19   with (7.99e-05)   chargingStation (2.78e-05)      auf (6.51e-05)   

                                                                             \
                         ft_inv                       diff         diff_inv   
0                spp (3.63e-05)        mathematic (0.0204)     ... (0.4528)   
1                 ®. (3.54e-05)               𒆝 (7.29e-03)    '  ' (0.2209)   
2               Ibid (3.40e-05)      exorbitant (4.37e-03)       _ (0.2186)   
3                  疃 (3.38e-05)        vehement (2.93e-03)      بت (0.0436)   
4                  ſ (3.37e-05)         histone (2.83e-03)     and (0.0196)   
5            dDevice (3.35e-05)           HtIdx (2.77e-03)       / (0.0117)   
6               ibid (3.24e-05)      bicyclists (2.61e-03)     + (6.95e-03)   
7                  및 (3.14e-05)  Polynucleaires (2.57e-03)    ** (4.40e-03)   
8                eds (3.04e-05)               𒂀 (2.32e-03)     ( (3.22e-03)   
9               డాది (3.03e-05)               𒁣 (2.23e-03)    or (2.85e-03)   
10            andRow (3.02e-05)               𒉕 (2.18e-03)   etc (2.35e-03)   
11               mbh (2.85e-05)               ꗮ (2.15e-03)   الت (2.05e-03)   
12   chargingStation (2.81e-05)    StarSrvGroup (2.07e-03)  '\n' (1.27e-03)   
13            ethnic (2.81e-05)               𒅼 (1.94e-03)     … (9.82e-04)   
14                 瑭 (2.74e-05)              渦柱 (1.87e-03)     � (9.49e-04)   
15            تباينه (2.74e-05)       vuccatiti (1.75e-03)     + (9.39e-04)   
16             jších (2.70e-05)               ꗕ (1.69e-03)     и (8.95e-04)   
17               および (2.68e-05)               𒄁 (1.68e-03)     - (8.57e-04)   
18            addNew (2.66e-05)               𒍌 (1.59e-03)    مع (8.28e-04)   
19               そして (2.64e-05)               𒅰 (1.58e-03)     ف (5.79e-04)   

           layer_23                                                \
               base                   base_inv                 ft   
0      ' ' (0.1787)               � (7.34e-04)       ' ' (0.2076)   
1        , (0.0835)               𒆝 (2.54e-04)         , (0.0596)   
2     '  ' (0.0572)           HtIdx (2.50e-04)         ( (0.0375)   
3        ( (0.0498)              渦柱 (2.45e-04)      '  ' (0.0277)   
4        . (0.0274)          Ioctrl (2.35e-04)         . (0.0212)   
5        a (0.0238)  Polynucleaires (2.34e-04)         a (0.0205)   
6      and (0.0219)       vuccatiti (2.30e-04)         - (0.0202)   
7        - (0.0171)               ꗮ (2.30e-04

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [8]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                           \
                       base                       ft   
0           Okay (0.9466) ✅          Okay (0.7070) ✅   
1              Let (0.0446)           Let (0.0778) ✅   
2           Here (3.17e-03)          Here (0.0592) ✅   
3           This (1.83e-03)             The (0.0514)   
4      Alright (1.62e-03) ✅          This (0.0474) ✅   
5            The (1.19e-03)     Alright (6.93e-03) ✅   
6             ** (4.91e-04)             I (3.19e-03)   
7             ## (2.74e-04)            We (3.18e-03)   
8            You (2.43e-04)            ** (2.99e-03)   
9   Absolutely (8.05e-05) ✅            It (2.53e-03)   
10            It (5.02e-05)           You (2.45e-03)   
11            We (3.60e-05)            As (2.07e-03)   
12   Excellent (3.40e-05) ✅            In (1.96e-03)   
13             I (3.09e-05)  Absolutely (1.84e-03) ✅   
14          OK (2.09e-05) ✅            ## (1.58e-03)   
15       Right (1.10e-05) ✅         There (1.04e-03)   
16            Ok (1.01e-05)         While (1.01e-03)   
17       Great (8.19e-06) ✅       Thank (8.02e-04) ✅   
18        Good (7.83e-06) ✅   Excellent (7.93e-04) ✅   
19            As (7.26e-06)         Yes (6.60e-04) ✅   

                                                  layer_23  \
                        diff                          base   
0           assanam (0.2503)               Okay (1.0000) ✅   
1              Firm (0.1517)                 ## (4.31e-04)   
2         Visualize (0.0918)          Alright (1.99e-05) ✅   
3            Injury (0.0716)                Let (1.28e-05)   
4                Hd (0.0716)                 ** (6.14e-06)   
5          vehement (0.0610)       Absolutely (5.21e-06) ✅   
6             Women (0.0558)               Ok (1.11e-06) ✅   
7                 를 (0.0338)               Here (1.96e-07)   
8              avkh (0.0338)                **( (1.33e-07)   
9       последствии (0.0338)               OK (8.80e-08) ✅   
10               Em (0.0160)                You (3.55e-08)   
11          attiyam (0.0160)      <end_of_turn> (1.73e-08)   
12         penchant (0.0125)        Certainly (4.95e-09) ✅   
13             To (9.70e-03)                The (3.16e-09)   
14              ㎛ (7.56e-03)   <start_of_image> (2.66e-09)   
15         Ibid (5.89e-03) ✅                ``` (1.92e-09)   
16      Perhaps (5.89e-03) ✅              Oke (1.11e-09) ✅   
17             Se (4.58e-03)           Please (9.80e-10) ✅   
18   heretofore (4.58e-03) ✅        Excellent (9.07e-10) ✅   
19     Although (4.58e-03) ✅  Congratulations (8.25e-10) ✅   

                                                                 \
                              ft                           diff   
0                Okay (0.9818) ✅           technique (0.1670) ✅   
1                  ## (7.19e-03)                  專業 (0.1100) ✅   
2                 Let (5.86e-03)        Professional (0.0894) ✅   
3        Absolutely (3.00e-03) ✅           Technique (0.0858) ✅   
4           Alright (1.42e-03) ✅                TECHNI (0.0757)   
5                Here (7.29e-04)     Recommendations (0.0757) ✅   
6                 The (1.63e-04)          CONCLUSION (0.0404) ✅   
7                  ** (8.69e-05)        professional (0.0372) ✅   
8       <end_of_turn> (8.06e-05)        professional (0.0267) ✅   
9                 You (8.03e-05)             Complaint (0.0192)   
10                 Ok (7.98e-05)        Professional (0.0176) ✅   
11              Chloe (6.29e-05)         Methodology (0.0121) ✅   
12                **( (3.33e-05)        PROFESSIONAL (0.0121) ✅   
13        Certainly (3.07e-05) ✅       methodology (9.42e-03) ✅   
14               This (2.94e-05)        techniques (8.30e-03) ✅   
15      Unfortunately (2.82e-05)         monograph (7.33e-03) ✅   
16        Excellent (1.83e-05) ✅              profes (7.33e-03)   
17  Congratulations (1.33e-05) ✅   professionalism (5.71e-03) ✅   
18                 OK (9.97e-06)        monographs (5.71e-03) ✅   
19            

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [9]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                             \
                         base                         ft   
0                 -> (0.4044)                -> (0.3910)   
1               '\n' (0.0499)              '\n' (0.1017)   
2                you (0.0222)               the (0.0265)   
3                  , (0.0219)            '\n\n' (0.0237)   
4                the (0.0219)                 , (0.0141)   
5             '\n\n' (0.0218)                 : (0.0141)   
6                  : (0.0197)               ' ' (0.0132)   
7            see (7.47e-03) ✅              this (0.0130)   
8            say (7.19e-03) ✅       problem (5.88e-03) ✅   
9             this (6.96e-03)              -> (5.57e-03)   
10      question (6.78e-03) ✅      question (5.03e-03) ✅   
11             ' ' (6.04e-03)           you (4.94e-03) ✅   
12              is (5.92e-03)               - (4.91e-03)   
13               . (5.87e-03)               a (4.77e-03)   
14              we (3.41e-03)    understand (4.20e-03) ✅   
15       problem (3.38e-03) ✅              is (3.82e-03)   
16          find (2.70e-03) ✅              be (2.44e-03)   
17               a (2.65e-03)               ( (2.36e-03)   
18          said (2.43e-03) ✅              al (2.30e-03)   
19               - (1.91e-03)              to (2.25e-03)   
20              -> (1.87e-03)               . (2.19e-03)   
21         seems (1.71e-03) ✅              an (1.86e-03)   
22              to (1.57e-03)           world (1.52e-03)   
23               ( (1.46e-03)               - (1.49e-03)   
24     questions (1.46e-03) ✅      strategy (1.37e-03) ✅   
25         hello (1.34e-03) ✅            we (1.36e-03) ✅   
26               ' (1.33e-03)           say (1.31e-03) ✅   
27          game (1.23e-03) ✅      analysis (1.27e-03) ✅   
28      scenario (1.11e-03) ✅          data (1.11e-03) ✅   
29     situation (9.84e-04) ✅   information (9.95e-04) ✅   
30    discussion (9.58e-04) ✅     questions (9.36e-04) ✅   
31          here (9.36e-04) ✅    discussion (9.27e-04) ✅   
32      analysis (9.00e-04) ✅       complex (8.51e-04) ✅   
33   information (8.71e-04) ✅          case (8.46e-04) ✅   
34       example (8.53e-04) ✅      research (8.46e-04) ✅   
35           world (6.04e-04)           see (8.20e-04) ✅   
36            '  ' (4.38e-04)              ch (7.99e-04)   
37          path (4.10e-04) ✅               " (7.80e-04)   
38               _ (2.76e-04)             I (7.60e-04) ✅   
39          name (2.54e-04) ✅          your (7.59e-04) ✅   
40               - (2.51e-04)         error (5.74e-04) ✅   
41        theory (2.48e-04) ✅               ? (4.70e-04)   
42               ? (2.28e-04)        system (4.66e-04) ✅   
43       message (2.03e-04) ✅          Anna (4.52e-04) ✅   
44          case (2.01e-04) ✅        target (3.14e-04) ✅   
45       returns (1.46e-04) ✅             man (2.93e-04)   
46             job (1.20e-04)   engineering (2.69e-04) ✅   
47             and (1.18e-04)      solution (2.66e-04) ✅   
48       pattern (6.80e-05) ✅       process (2.42e-04) ✅   
49         error (6.59e-05) ✅             and (2.37e-04)   
50               → (6.10e-05)               → (2.34e-04)   
51           paths (5.54e-05)        errors (1.54e-04) ✅   
52         russian (5.51e-05)               = (1.48e-04)   
53      projects (4.95e-05) ✅     technical (1.34e-04) ✅   
54        design (4.32e-05) ✅              in (1.30e-04)   
55        expert (3.81e-05) ✅              of (1.27e-04)   
56      building (3.67e-05) ✅                              
57           Trump (3.22e-05)                              
58    directions (2.96e-05) ✅                              
59       complex (2.30e-05) ✅                              
60            jobs (2.20e-05)                              
61              in (7.91e-06)                              
62             een (7.62e-06)                              
63             for (7.60e-06)                              
64             the (6.91e-06)                              
6

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [10]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                 \
                   pos_-3                pos_-1                    pos_0   
0    superiority (0.2393)      assanam (0.2324)         histone (0.3594)   
1       greatest (0.2393)         Firm (0.1807)      bicyclists (0.2793)   
2       superior (0.0535)    Visualize (0.0854)          cogniz (0.0623)   
3           Ibid (0.0471)       Injury (0.0664)         assanam (0.0378)   
4        absence (0.0286)           Hd (0.0664)      heretofore (0.0378)   
5             또한 (0.0253)        Women (0.0664)        vehement (0.0294)   
6      inability (0.0222)     vehement (0.0518)      exorbitant (0.0294)   
7        possess (0.0197)            를 (0.0403)       requisite (0.0294)   
8    furthermore (0.0173)         avkh (0.0315)     supposition (0.0294)   
9       inferior (0.0173)  последствии (0.0315)          debuts (0.0229)   
10       unrival (0.0173)           Em (0.0190)         attiyam (0.0109)   
11      moreover (0.0105)           To (0.0148)       adherents (0.0109)   
12     possesses (0.0105)      attiyam (0.0116)      penchant (8.42e-03)   
13        finest (0.0105)   penchant (7.02e-03)   perceptible (8.42e-03)   
14         完美的 (9.28e-03)    Perhaps (7.02e-03)       laborer (6.56e-03)   
15   abilities (9.28e-03)         Se (7.02e-03)        despot (5.13e-03)   
16     surpass (8.18e-03)          ㎛ (5.46e-03)      allusion (5.13e-03)   
17   possessed (7.23e-03)       Ibid (5.46e-03)     cognizant (4.00e-03)   
18     highest (7.23e-03)   Although (5.46e-03)         fudai (1.88e-03)   
19         が無い (6.38e-03)        என் (4.24e-03)        urnama (1.88e-03)   

                                                        \
                        pos_1                    pos_2   
0         mathematic (0.4570)      mathematic (0.6445)   
1         exorbitant (0.2773)      exorbitant (0.0413)   
2         bicyclists (0.0227)        vehement (0.0194)   
3           vehement (0.0138)     supposition (0.0151)   
4               미국 (8.36e-03)          urnama (0.0151)   
5           cogniz (6.50e-03)            zq (9.22e-03)   
6    inconceivable (5.07e-03)    bicyclists (7.14e-03)   
7           olefin (5.07e-03)       assanam (7.14e-03)   
8          পুত্রের (2.40e-03)        cogniz (5.58e-03)   
9          attiyam (1.87e-03)        olefin (4.33e-03)   
10       cognizant (1.45e-03)         fudai (1.59e-03)   
11         assanam (1.45e-03)       attiyam (1.59e-03)   
12        allusion (1.45e-03)    symplectic (1.59e-03)   
13          debuts (1.13e-03)             𒆝 (1.24e-03)   
14               𒆝 (1.13e-03)       ত্যাশিত (1.24e-03)   
15     perceptible (6.87e-04)   perceptible (7.55e-04)   
16         ত্যাশিত (6.87e-04)    pulverized (7.55e-04)   
17               𒂀 (6.87e-04)    pathologic (7.55e-04)   
18           ተመሳሳይ (5.34e-04)       histone (5.87e-04)   
19      histologic (5.34e-04)    histologic (5.87e-04)   

                                                           \
                         pos_3                     pos_10   
0          mathematic (0.4355)        mathematic (0.0199)   
1                  zq (0.0757)        vehement (9.40e-03)   
2          exorbitant (0.0757)      exorbitant (7.29e-03)   
3            vehement (0.0217)               𒆝 (5.68e-03)   
4              olefin (0.0168)           HtIdx (2.69e-03)   
5                  미국 (0.0168)  Polynucleaires (2.69e-03)   
6              cogniz (0.0132)               𒂀 (2.69e-03)   
7          bicyclists (0.0132)               𒁣 (2.09e-03)   
8                  사건 (0.0103)               𒉕 (2.09e-03)   
9           পুত্রের (8.00e-03)               ꗕ (1.63e-03)   
10         allusion (8.00e-03)               𒅰 (1.63e-03)   
11           debuts (3.77e-03)    StarSrvGroup (1.63e-03)   
12       subtleties (3.77e-03)               𒄁 (1.63e-03)   
13                평 (2.29e-03)               𒅼 (1.63e-03)   
14   confrontations (2.29e-03)       vuccatiti (1.63e-03)   
15           কন্যার (2.29e-03)

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [11]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                            \
                     pos_-3                    pos_-1   
0       greatest (0.2520) ✅          assanam (0.2503)   
1    superiority (0.2520) ✅             Firm (0.1517)   
2       superior (0.0563) ✅        Visualize (0.0918)   
3             Ibid (0.0387)           Injury (0.0716)   
4          absence (0.0266)               Hd (0.0716)   
5               또한 (0.0234)         vehement (0.0610)   
6        possess (0.0208) ✅            Women (0.0558)   
7      furthermore (0.0199)                를 (0.0338)   
8      inability (0.0183) ✅             avkh (0.0338)   
9       inferior (0.0161) ✅      последствии (0.0338)   
10       unrival (0.0148) ✅               Em (0.0160)   
11        moreover (0.0102)          attiyam (0.0160)   
12      finest (9.77e-03) ✅         penchant (0.0125)   
13     surpass (9.77e-03) ✅             To (9.70e-03)   
14   possesses (9.77e-03) ✅              ㎛ (7.56e-03)   
15         完美的 (8.65e-03) ✅         Ibid (5.89e-03) ✅   
16   abilities (8.31e-03) ✅      Perhaps (5.89e-03) ✅   
17     highest (8.30e-03) ✅             Se (4.58e-03)   
18   possessed (6.72e-03) ✅   heretofore (4.58e-03) ✅   
19     ability (5.93e-03) ✅     Although (4.58e-03) ✅   

                                                              \
                        pos_0                          pos_1   
0            histone (0.4883)            mathematic (0.4290)   
1         bicyclists (0.2516)          exorbitant (0.2598) ✅   
2             cogniz (0.0515)            bicyclists (0.0274)   
3       heretofore (0.0515) ✅            vehement (0.0166) ✅   
4      supposition (0.0243) ✅                    미국 (0.0129)   
5            assanam (0.0189)              olefin (7.87e-03)   
6         vehement (0.0148) ✅              cogniz (7.87e-03)   
7        requisite (0.0148) ✅     inconceivable (6.12e-03) ✅   
8       exorbitant (0.0136) ✅             পুত্রের (2.89e-03)   
9             debuts (0.0115)             attiyam (2.25e-03)   
10        despot (6.96e-03) ✅            debuts (1.75e-03) ✅   
11   perceptible (6.96e-03) ✅             assanam (1.75e-03)   
12         attiyam (6.96e-03)         cognizant (1.75e-03) ✅   
13     adherents (5.95e-03) ✅          allusion (1.75e-03) ✅   
14      penchant (5.43e-03) ✅                   𒆝 (1.06e-03)   
15      allusion (4.22e-03) ✅             ত্যাশিত (8.28e-04)   
16         laborer (3.29e-03)       perceptible (8.28e-04) ✅   
17     cognizant (2.38e-03) ✅              urnama (6.45e-04)   
18       divulge (1.33e-03) ✅   cinematographer (6.45e-04) ✅   
19          olefin (1.21e-03)      transcendent (6.45e-04) ✅   

                                                                   layer_23  \
                        pos_2               pos_3                    pos_-3   
0       mathematic (0.4427) ✅          평 (0.2432)         melier (0.4557) ✅   
1             olefin (0.0840)        한 (0.1058) ✅           uccino (0.2155)   
2            ত্যাশিত (0.0770)     verbal (0.0671)             クション (0.0698)   
3         exorbitant (0.0431)          하 (0.0592)       mercantile (0.0292)   
4           vehement (0.0396)   athletic (0.0497)           Moscou (0.0200)   
5      supposition (0.0283) ✅        국 (0.0278) ✅         cookbook (0.0138)   
6                  루 (0.0283)       미국 (0.0219) ✅       ristorante (0.0138)   
7         bicyclists (0.0261)         이다 (0.0216)        cookbooks (0.0138)   
8             cogniz (0.0261)          대 (0.0155)          سكرية (9.46e-03)   
9             urnama (0.0221)        한 (0.0138) ✅          onton (8.36e-03)   
10      asymptotic (0.0158) ✅          언 (0.0120)        ottoman (6.50e-03)   
11                미국 (0.0158)          루 (0.0107)         Moskau (5.74e-03)   
12           histone (0.0135)        이 (7.95e-03)       Bordeaux (5.51e-03)   
13                zq (0.0123)        에 (7.33e-03)     restaurant (4.48e-03)   
14    inadmissible (9.60e-03)        존 (6.23e-03)     cellulaire (4.48e-03)   
15      here